# 第23章 数据合并与结构转换

通过merge、concat、melt、stack和unstack整合并重塑表格。

## 本章定位

本章按“概念 → 示例 → 练习”的顺序组织。代码单元格可以单独运行，也可以从上到下完整运行。

## 学习目标

- 按键连接表格
- 纵向与横向拼接
- 宽表转长表
- 在索引层级间重塑


## 核心概念

- 连接前检查键唯一性和匹配率。
- concat负责轴向拼接，merge负责关系连接。
- 长表更适合分组统计和可视化。


## 示例 1：关系连接

validate参数可以验证预期的一对一或多对一关系。


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "order_id": ["A1", "A2", "A3"],
    "customer_id": ["U1", "U2", "U1"],
    "amount": [320, 880, 460],
})
customers = pd.DataFrame({
    "customer_id": ["U1", "U2"],
    "city": ["上海", "广州"],
})
merged = orders.merge(customers, on="customer_id", how="left", validate="many_to_one", indicator=True)
print(merged)


## 示例 2：数据拼接

拼接后通常需要重新建立连续索引。


In [ ]:
january = pd.DataFrame({"month": ["1月", "1月"], "sales": [120, 98]})
february = pd.DataFrame({"month": ["2月", "2月"], "sales": [150, 132]})
combined = pd.concat([january, february], ignore_index=True)
print(combined)


## 示例 3：宽表转长表

melt明确保留标识列，把多个指标列折叠为变量和值。


In [ ]:
wide = pd.DataFrame({
    "region": ["华东", "华南"],
    "一月": [120, 98],
    "二月": [150, 132],
    "三月": [180, 145],
})
long = wide.melt(id_vars="region", var_name="month", value_name="sales")
restored = long.pivot(index="region", columns="month", values="sales")
print(long)
print(restored)


## 初学者学习路线

这章建议按照“先观察、再模仿、后修改、最后独立完成”的顺序学习，不必一次记住所有参数。

1. 先阅读任务说明，明确这段代码要回答什么问题。
2. 运行一个最小例子，先观察输入、输出和数据形状，再回看每一行代码。
3. 只修改一个参数或一条数据，重新运行并比较前后结果。
4. 完成“综合练习”，最后再看本章小结，把能迁移到其他数据的问题写下来。

运行时如果看到 NameError，通常是前置单元格还没有运行；如果输出和预期不同，先检查变量是否被后面的单元格重新赋值。


## 先做一个小检查

进入正式例子前，先用一句话回答：本章的输入是什么，想得到什么结果？

本章主题是“第23章 数据合并与结构转换”。请特别留意三件事：输入的类型或形状、处理中间变量的含义、最后输出能否支持一个清楚的结论。


## Pandas 的学习主线

写数据字典 → 读取与检查 → 清洗类型和缺失值 → 选择与筛选 → 新增计算列 → 分组聚合 → 合并或透视 → 导出可复用结果

每一步都说明“一行代表什么”。处理前后记录行数、列数和关键字段；汇总前先确认分组粒度，避免得到数字却无法解释。


## 本模块练习方式

基础：完成一个字段清洗；提高：从明细表生成汇总表；挑战：处理重复、缺失和类型混乱，并写出清洗规则。

完成后请写下：输入是什么、处理做了什么、输出说明了什么、还存在什么限制。


## 示例 4：从明细表生成计算列

这一组例子只处理一个小问题。先运行代码，再逐行对照拆解说明。


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "region": ["华东", "华南", "华东"],
    "sales": [120, 150, 180],
    "cost": [80, 100, 130],
})
orders["profit"] = orders["sales"] - orders["cost"]
orders["profit_rate"] = orders["profit"] / orders["sales"]
print(orders.round(3))


### 逐步拆解

先新增一个简单指标，再基于它计算比例；拆成多列可以保留中间结果并方便检查。

建议第一次运行后只改一个输入值，再观察哪一个输出发生变化。


## 示例 5：从明细汇总到业务表

看懂上一个例子后，再观察同一主题在另一种数据或场景中的写法。


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "region": ["华东", "华东", "华南", "华南"],
    "channel": ["线上", "线下", "线上", "线下"],
    "sales": [120, 80, 150, 100],
})
summary = orders.groupby(["region", "channel"], as_index=False)["sales"].sum()
print(summary)
print("地区合计：")
print(orders.groupby("region")["sales"].sum())


### 逐步拆解

先明确每一行的粒度，再选择 groupby 的字段；汇总表的每一行代表一个清晰的分组组合。

自我检查：如果把输入数量、类别或参数改成另一组值，代码是否仍然能运行？


## 教学实验：分组汇总与粒度

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "region": ["华东", "华东", "华南", "华南"],
    "channel": ["线上", "线下", "线上", "线下"],
    "sales": [120, 80, 150, 100],
})
summary = orders.groupby("region", as_index=False)["sales"].sum()
print(summary)
print("汇总表每一行代表一个地区")


### 第一个结果怎么读

先确认明细表一行代表一笔订单，再确认汇总表一行代表一个地区。`groupby` 的字段决定结果的粒度。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
orders["sales_level"] = orders["sales"].map(
    lambda value: "高" if value >= 120 else "普通"
)
print(orders)
print(orders["sales_level"].value_counts())


### 第二个结果怎么读

第二个实验只增加一个分类列，不改变原始销售额。练习解释：什么时候应该新增列，什么时候应该直接筛选行？

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：脏数据转换怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

raw = pd.Series(["12", "unknown", "18", ""])
converted = pd.to_numeric(raw, errors="coerce")
print("转换结果：")
print(converted)
print("无法转换的数量：", converted.isna().sum())
print("后续可以选择删除、填充或回查原始值。")


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

errors="coerce" 会把无法转换的值记录为缺失，适合先完成质量盘点；不要在没有统计数量前直接删除。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 常见误区

- 连接键不唯一导致行数意外膨胀
- 连接后不检查未匹配记录
- 重塑时遗漏标识列


## 综合练习

1. 连接订单表与商品表
2. 计算订单行金额
3. 将月度宽表转换为长表

请先独立完成，再点击下方“显示答案”查看参考代码。


In [ ]:
import pandas as pd

items = pd.DataFrame({"product_id": ["P1", "P2"], "price": [299, 129]})
order_lines = pd.DataFrame({
    "order_id": ["A1", "A1", "A2"],
    "product_id": ["P1", "P2", "P2"],
    "quantity": [1, 2, 3],
})

# TODO: 连接订单表与商品表
result = order_lines.merge(items, on="product_id", validate="many_to_one")

# TODO: 计算订单行金额
result["line_amount"] = result["quantity"] * result["price"]

print(result)
print(result.groupby("order_id")["line_amount"].sum())


In [ ]:
import pandas as pd

items = pd.DataFrame({"product_id": ["P1", "P2"], "price": [299, 129]})
order_lines = pd.DataFrame({
    "order_id": ["A1", "A1", "A2"],
    "product_id": ["P1", "P2", "P2"],
    "quantity": [1, 2, 3],
})
result = order_lines.merge(items, on="product_id", validate="many_to_one")
result["line_amount"] = result["price"] * result["quantity"]
print(result)
print(result.groupby("order_id")["line_amount"].sum())

# 自检
assert len(result) == 3, "检查连接结果：应该有3行"
assert result["line_amount"].sum() == 686, "检查总金额：299 + 258 + 387"


## 本章小结

通过merge、concat、melt、stack和unstack整合并重塑表格。

**迁移思考**：

1. 如果订单表和商品表连接后行数突然增加了10倍，最可能的原因是什么？如何诊断？
2. 为什么长表更适合分组统计和可视化？宽表适合什么场景？


### 你已经掌握

- 按键连接表格
- 纵向与横向拼接
- 宽表转长表
- 在索引层级间重塑


### 关键知识速查

| 知识点 | 作用与提醒 | 关键写法 |
| --- | --- | --- |
| 关系连接 | validate参数可以验证预期的一对一或多对一关系。 | `pd.DataFrame()`、`orders.merge()` |
| 数据拼接 | 拼接后通常需要重新建立连续索引。 | `pd.DataFrame()`、`pd.concat()` |
| 宽表转长表 | melt明确保留标识列，把多个指标列折叠为变量和值。 | `pd.DataFrame()`、`wide.melt()`、`long.pivot()` |


### 需要注意

- 连接键不唯一导致行数意外膨胀
- 连接后不检查未匹配记录
- 重塑时遗漏标识列


### 完成检查

- [ ] 能够按键连接表格
- [ ] 能够纵向与横向拼接
- [ ] 能够宽表转长表
- [ ] 能够在索引层级间重塑
